# Tesseract OCR Inference (Kaggle)
This notebook downloads the AAR rulings image dataset from Hugging Face and runs PyTesseract to extract text, benchmarking the performance.

In [ ]:
!pip install pytesseract datasets pandas tqdm pillow

In [ ]:
import time
import pandas as pd
import pytesseract
from datasets import load_dataset
from tqdm.auto import tqdm
import os

# Ensure tesseract is installed in the Kaggle environment (apt-get if needed)
!apt-get update
!apt-get install -y tesseract-ocr

In [ ]:
# Load dataset from Hugging Face
# Replace 'your_hf_username' with your actual username/dataset location
dataset_name = "your_hf_username/aar_rulings_ocr_sample"
print(f"Loading dataset: {dataset_name}")
try:
    dataset = load_dataset(dataset_name, split="train")
    print(f"Loaded {len(dataset)} images.")
except Exception as e:
    print(f"Error loading dataset: {e}")
    print("Make sure the dataset is public, or that you've logged in with your HF Token.")

In [ ]:
# Run inference
results = []
start_time_total = time.time()

if 'dataset' in locals():
    for idx, item in enumerate(tqdm(dataset, desc="Running Tesseract OCR")):
        img = item["image"]
        pdf_name = item["pdf_name"]
        page_num = item["page_num"]
        
        start_time = time.time()
        error = None
        text = ""
        try:
            text = pytesseract.image_to_string(img)
        except Exception as e:
            error = str(e)
            
        runtime = time.time() - start_time
        
        results.append({
            "pdf_name": pdf_name,
            "page_num": page_num,
            "runtime_seconds": runtime,
            "extracted_text_length": len(text),
            "error": error
        })

    total_runtime = time.time() - start_time_total
    df_results = pd.DataFrame(results)
else:
    print("Dataset not loaded, skipping inference.")

In [ ]:
# Metrics & Saving
if 'df_results' in locals() and not df_results.empty:
    avg_speed = df_results["runtime_seconds"].mean()
    errors_count = df_results["error"].notnull().sum()

    print(f"Total Runtime: {total_runtime:.2f} seconds")
    print(f"Average Speed: {avg_speed:.4f} seconds/page")
    print(f"Total Errors: {errors_count}")

    output_file = "/kaggle/working/tesseract_results.csv"
    df_results.to_csv(output_file, index=False)
    print(f"Results saved to {output_file}")
